<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

# 학습 내용
>이번 장에서는 <strong>Agent 4 Components(에이전트의 4가지 구성요소)</strong>에 대해 학습합니다.
>Tool, Tool Binding, ToolNode, Memory, create_react_agent를 하나씩 추가하며 Agent를 학습해봅시다.

# Agent의 4가지 구성요소
> Agent는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">LLM + Tool + Planning + Memory</mark>로 구성됩니다.

<strong>AI Agent</strong>란 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">LLM + 도구(Tool) + 메모리(Memory) + 계획(Planning)</mark>이 결합된 자율 실행 시스템입니다. 단순한 LLM 호출과 Agent의 결정적 차이는 <strong>자율성</strong>에 있습니다. <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">단순 LLM 호출</mark>은 질문을 받으면 그 자리에서 답변을 생성하고 종료하지만, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Agent</mark>는 목표를 달성하기 위해 스스로 어떤 도구를 사용할지 판단하고, 도구를 실행하고, 결과를 바탕으로 다음 행동을 결정합니다. 예를 들어 "환불 가능한지 알려줘"라는 질문에 단순 LLM은 학습 데이터 기반으로 추측하지만, Agent는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">실제 환불 정책 DB를 검색하고 정확한 답을 제공</mark>합니다.</br>
이 내용을 학습하기 전에 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">LLM API</mark>(OpenAI, Anthropic 등의 API 호출 방식), <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">함수 호출(Function Calling)</mark>(LLM이 외부 함수를 호출하도록 지시하는 메커니즘), <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">RAG</mark>(외부 문서를 검색하여 LLM의 답변 품질을 높이는 기법)의 개념을 먼저 이해하면 좋습니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">구성요소</th>
      <th>역할</th>
      <th>예시</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">LLM</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">판단과 생성</mark></td><td>GPT-4o, Claude</td></tr>
    <tr><td style="text-align:center">Tool</td><td>외부 기능 실행</td><td>검색, 계산, API 호출</td></tr>
    <tr><td style="text-align:center">Planning</td><td>작업 계획 수립</td><td>ReAct, Direct</td></tr>
    <tr><td style="text-align:center">Memory</td><td>대화 기록 유지</td><td>상태 관리</td></tr>
  </tbody>
</table>

# Tool 정의 (@tool 데코레이터)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">@tool 데코레이터</mark>로 Python 함수를 LLM이 호출할 수 있는 도구로 변환합니다.

In [ ]:
# TODO 1: 도구 데코레이터로 search_policy 함수를 정의하고 (입력: query: str, 반환: str), docstring에 "고객 서비스 정책을 검색합니다."를 작성한 뒤, 검색기로 관련 문서를 검색하여 결과를 결합하여 반환하세요. 마지막으로 도구의 이름, 설명, 입력 스키마를 출력하세요.

from langchain_core.tools import tool

@tool
def search_policy(query: str) -> str:
    """고객 서비스 정책을 검색합니다."""
    documents = retriever.invoke(query)
    return "\n".join([doc.page_content for doc in documents])

print(f"도구 이름: {search_policy.name}")
print(f"도구 설명: {search_policy.description}")
print(f"입력 스키마: {search_policy.args}")

💡docstring의 중요성
> LLM은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">docstring을 읽고</mark> 어떤 도구를 사용할지 판단합니다.
> 명확하고 구체적인 설명이 도구 선택 정확도를 높입니다.

# Tool Binding (bind_tools)
> LLM에 사용 가능한 도구 목록을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">바인딩</mark>합니다.

In [ ]:
# TODO 2: tools 리스트에 search_policy를 담고, LLM에 도구를 바인딩하여 llm_with_tools를 생성하세요. "환불 정책이 뭐야?"를 전달한 뒤, 응답의 도구 호출 수와 내용을 출력하세요.

tools = [search_policy]
llm_with_tools = llm.bind_tools(tools)

# LLM이 도구 호출을 결정
response = llm_with_tools.invoke("환불 정책이 뭐야?")

# tool_calls 확인
print(f"tool_calls 수: {len(response.tool_calls)}")
print(f"tool_calls: {response.tool_calls}")

## tool_calls 구조

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">필드</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>name</code></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">호출할 도구 이름</mark></td></tr>
    <tr><td style="text-align:center"><code>args</code></td><td>도구에 전달할 인자</td></tr>
    <tr><td style="text-align:center"><code>id</code></td><td>호출 식별자</td></tr>
  </tbody>
</table>

# ToolNode (도구 실행기)
> LLM이 요청한 도구를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">실제로 실행</mark>하는 노드입니다.

In [ ]:
# TODO 3: 도구 실행 노드를 생성하고 등록된 도구 이름 리스트를 출력하세요.

from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)
print(f"등록된 도구: {[t.name for t in tools]}")

💡bind_tools vs ToolNode
> `bind_tools`: LLM이 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">어떤 도구를 쓸지 결정</mark>하게 함 (도구 목록 제공)
> `ToolNode`: LLM의 결정에 따라 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">실제로 도구를 실행</mark> (도구 실행기)

💡왜 분리되어 있는가?
> LLM의 "판단"과 "실행"을 분리하면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">안전성 검증</mark>을 중간에 삽입할 수 있습니다.
> 예: LLM이 삭제 API를 호출하려 할 때 사람의 승인을 먼저 받는 구조.

# Memory (MemorySaver)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">MemorySaver</mark>로 대화 기록을 저장하여 Agent가 맥락을 유지합니다.

Agent는 기본적으로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">각 호출이 독립적</mark>입니다. "내 주문번호는 ORD001이야"라고 말한 뒤 "아까 말한 주문 상태가 뭐야?"라고 물으면, Agent는 이전 대화를 기억하지 못합니다. <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">MemorySaver</mark>는 LangGraph의 체크포인터(checkpointer)로, 대화 메시지를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">thread_id별로 저장</mark>하여 동일 스레드 내에서 맥락을 유지합니다. 같은 `thread_id`를 사용하면 이전 대화가 자동으로 이어지고, 다른 `thread_id`를 사용하면 새로운 대화가 시작됩니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">구성요소</th>
      <th>설명</th>
      <th>예시</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>MemorySaver</code></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">인메모리 체크포인터</mark></td><td>개발/테스트용 대화 기록 저장</td></tr>
    <tr><td style="text-align:center"><code>checkpointer</code></td><td>Agent 생성 시 전달하는 메모리 객체</td><td><code>create_react_agent(..., checkpointer=memory)</code></td></tr>
    <tr><td style="text-align:center"><code>thread_id</code></td><td>대화 세션 식별자</td><td><code>{"configurable": {"thread_id": "user123"}}</code></td></tr>
  </tbody>
</table>

In [ ]:
# TODO 4: MemorySaver를 생성하세요.

from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
print(f"체크포인터 타입: {type(memory).__name__}")

In [ ]:
# TODO 5: checkpointer를 전달하여 Memory가 적용된 Agent를 생성하고, 동일한 thread_id로 두 번 호출하여 맥락 유지를 확인하세요.

from langgraph.prebuilt import create_react_agent

agent_with_memory = create_react_agent(
    model=llm,
    tools=tools,
    checkpointer=memory
)

config = {"configurable": {"thread_id": "user123"}}

# 첫 번째 대화
response1 = agent_with_memory.invoke(
    {"messages": [("user", "내 주문번호는 ORD001이야.")]},
    config=config
)
print("응답 1:", response1["messages"][-1].content)

# 두 번째 대화 — 같은 thread_id로 맥락 유지
response2 = agent_with_memory.invoke(
    {"messages": [("user", "아까 말한 주문번호가 뭐였지?")]},
    config=config
)
print("응답 2:", response2["messages"][-1].content)

💡thread_id의 역할
> 같은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">thread_id</mark>를 사용하면 이전 대화가 이어지고, 다른 thread_id를 사용하면 새 대화가 시작됩니다.
> 실제 서비스에서는 사용자 세션 ID를 thread_id로 활용합니다.

# create_react_agent (프리빌트 Agent)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">create_react_agent</mark>로 LLM + Tool + Memory를 결합한 ReAct Agent를 한 줄로 생성합니다.

`create_react_agent`는 LangGraph가 제공하는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">프리빌트(prebuilt) Agent 생성 함수</mark>입니다. 앞서 배운 Tool 바인딩, ToolNode, MemorySaver를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">내부적으로 조합</mark>하여 ReAct 패턴(Thought → Action → Observation 순환)이 자동으로 동작하는 Agent를 생성합니다. StateGraph를 직접 구성하지 않아도 되므로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">빠른 프로토타이핑</mark>에 적합합니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">파라미터</th>
      <th>설명</th>
      <th>예시</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>model</code></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">LLM 인스턴스</mark></td><td><code>ChatOpenAI(model="gpt-4o-mini")</code></td></tr>
    <tr><td style="text-align:center"><code>tools</code></td><td>Agent가 사용할 도구 리스트</td><td><code>[search_policy, issue_coupon]</code></td></tr>
    <tr><td style="text-align:center"><code>prompt</code></td><td>시스템 프롬프트 (역할 지정)</td><td><code>"당신은 고객 서비스 Agent입니다."</code></td></tr>
    <tr><td style="text-align:center"><code>checkpointer</code></td><td>대화 기록 저장 (Memory)</td><td><code>MemorySaver()</code></td></tr>
  </tbody>
</table>

In [ ]:
# TODO 6: create_react_agent로 시스템 프롬프트와 checkpointer를 포함한 Agent를 생성하세요.

from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

system_prompt = "당신은 고객 서비스 AI 에이전트입니다. 고객의 요청을 처리하기 위해 제공된 도구를 사용하세요."

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt,
    checkpointer=MemorySaver()
)

print(f"Agent 타입: {type(agent).__name__}")

In [ ]:
# TODO 7: 생성한 Agent에 thread_id를 지정하여 멀티스텝 요청을 실행하고 최종 응답을 출력하세요.

config = {"configurable": {"thread_id": "demo-thread"}}

response = agent.invoke(
    {"messages": [("user", "환불 정책이 뭐야?")]},
    config=config
)
print("최종 응답:")
print(response["messages"][-1].content)

💡create_react_agent vs StateGraph 직접 구현
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">create_react_agent</mark>는 프로토타이핑에 적합하고, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">StateGraph 직접 구현</mark>은 커스텀 워크플로우에 적합합니다.
> 다음 노트북(Ch.4-2-1_002)에서 StateGraph를 직접 구현하여 내부 동작 원리를 학습합니다.